# Week 5 — Spark Fundamentals: Data Cleaning, Transformation & Aggregation

**Objective:** Understand Spark fundamentals and perform data cleaning, transformation, and
aggregation using DataFrames.

This notebook works through all 15 questions of the Week 5 assignment using PySpark,
running against a synthetic `dataset.csv` (1,050 rows, including intentional duplicates,
nulls, and empty strings to exercise every cleaning operation asked about).


In [1]:
# Setup: create Spark session and load the dataset
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

spark = SparkSession.builder \
    .appName("Week5_Spark_Assignment") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("../data/dataset.csv", header=True, inferSchema=True)
print("Row count:", df.count())
print("Columns:", df.columns)
df.printSchema()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/16 08:18:56 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/16 08:18:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/usr/local/lib/python3.12/dist-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/07/16 08:18:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Row count: 1050
Columns: ['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'store_id']
root
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)



In [2]:
df.show(5, truncate=False)

+-------+----------------+------+----------------+-----------+--------+-------+---+------------+-------------------+-----------------+--------+------+--------+
|user_id|transaction_date|region|product_category|sale_amount|status  |city   |age|subscription|raw_timestamp      |email            |username|price |store_id|
+-------+----------------+------+----------------+-----------+--------+-------+---+------------+-------------------+-----------------+--------+------+--------+
|U1001  |2025-06-13      |West  |Electronics     |1485.69    |Inactive|Houston|23 |Free        |2025-01-27 21:47:00|user1@example.com|user_1  |19.75 |STORE_2 |
|U1002  |2025-03-01      |West  |Toys            |405.69     |NULL    |Houston|43 |Free        |2025-03-13 00:48:00|user2@example.com|user_2  |81.96 |STORE_3 |
|U1003  |2025-01-27      |West  |Furniture       |202.47     |Pending |Denver |31 |Premium     |2025-07-06 14:34:00|user3@example.com|user_3  |150.12|STORE_5 |
|U1004  |2025-04-03      |East  |Electro

## Q1: Key limitations of traditional MapReduce that make Spark preferred

Traditional MapReduce has several limitations that Spark addresses:

1. **Disk I/O between stages** — MapReduce writes intermediate results (Map output, shuffle
   data, Reduce output) to disk (HDFS) after every stage. This is slow, especially for
   multi-step or iterative jobs.
2. **No in-memory computation** — every job re-reads and re-writes data from/to disk, which
   is a major bottleneck for iterative algorithms (e.g., machine learning, graph processing).
3. **Rigid two-stage model** — MapReduce only supports Map → Reduce. Any complex pipeline
   (multiple transformations, joins, iterative loops) needs to be broken into a chain of
   separate MapReduce jobs, each with its own disk I/O overhead.
4. **High latency for small/interactive jobs** — job startup and disk overhead make MapReduce
   poorly suited for interactive queries or ad hoc analysis.
5. **Limited high-level APIs** — MapReduce requires writing verbose low-level Java code for
   even simple operations; there is no built-in DataFrame/SQL abstraction.
6. **No built-in support for streaming or SQL** — these require separate tools (e.g., Hive,
   Storm) bolted on top of MapReduce.

Spark solves these by keeping data **in memory** across operations (RDD/DataFrame caching),
supporting a **rich DAG execution engine** that can chain many transformations without
writing to disk in between, and offering a **unified API** (batch, SQL, streaming, ML, graph)
on top of DataFrames — making it 10–100x faster for iterative and interactive workloads.

## Q2: In-Memory Computing and iterative ML algorithms

Iterative ML algorithms (e.g., gradient descent, k-means, PageRank) repeatedly scan and
transform the *same* dataset across many iterations.

- **Disk-based systems (classic MapReduce):** each iteration is a separate MapReduce job.
  The dataset must be read from disk, processed, and the result written back to disk before
  the next iteration starts. With N iterations, the dataset is read/written from disk N times
  — this I/O dominates runtime.
- **Spark's in-memory model:** Spark can `cache()` or `persist()` a DataFrame/RDD in memory
  (across the cluster's executors) after it is first computed. Every subsequent iteration
  reads directly from memory instead of disk, avoiding repeated disk I/O.

Because memory access is orders of magnitude faster than disk access, and Spark's DAG
scheduler avoids materializing intermediate results unless asked to, iterative algorithms
that would take many disk round-trips in MapReduce can run **10-100x faster** in Spark.

```python
# Example: caching a DataFrame used across iterations
df_features = spark.read.csv("features.csv", header=True, inferSchema=True)
df_features.cache()          # materializes into memory on first action
df_features.count()          # triggers the cache to populate

for i in range(10):
    # each iteration reuses the in-memory df_features instead of re-reading from disk
    result = df_features.groupBy("cluster_id").agg(F.avg("value"))
    result.show()
```

## Q3: Remove duplicate rows based on `user_id` and `transaction_date`

In [3]:
df_dedup = df.dropDuplicates(["user_id", "transaction_date"])

print("Original row count:", df.count())
print("After dropDuplicates(['user_id','transaction_date']):", df_dedup.count())
df_dedup.show(5)


Original row count: 1050


After dropDuplicates(['user_id','transaction_date']): 995


+-------+----------------+------+----------------+-----------+--------+-------+---+------------+-------------------+-------------------+--------+------+--------+
|user_id|transaction_date|region|product_category|sale_amount|  status|   city|age|subscription|      raw_timestamp|              email|username| price|store_id|
+-------+----------------+------+----------------+-----------+--------+-------+---+------------+-------------------+-------------------+--------+------+--------+
|  U1000|      2025-03-31| South|       Furniture|    1693.07| Pending| Dallas| 53|        Free|2025-02-02 16:06:00|user400@example.com|user_400|280.29| STORE_1|
|  U1000|      2025-07-07| South|            Toys|      261.1|    NULL|Phoenix| 22|        Free|2025-06-30 03:28:00|user800@example.com|user_800| 72.72| STORE_3|
|  U1001|      2025-03-19| South|       Furniture|    1185.17|Inactive| Denver| 30|        Free|2025-02-10 15:24:00|user401@example.com|user_401|218.42| STORE_4|
|  U1001|      2025-06-13|  

## Q4: Filter region == 'West', then group by `product_category` for average `sale_amount`

In [4]:
df_west_avg = (
    df_sales := df  # alias to match the question's naming
) \
    .filter(F.col("region") == "West") \
    .groupBy("product_category") \
    .agg(F.avg("sale_amount").alias("avg_sale_amount"))

df_west_avg.orderBy(F.desc("avg_sale_amount")).show()


+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|        Clothing|1134.4644444444443|
|       Furniture|1091.3509803921565|
|     Electronics|1062.5891999999994|
|       Groceries| 1043.126170212766|
|            Toys|1018.8356363636361|
+----------------+------------------+



## Q5: `.na.drop()` vs `.na.fill()`

- **`.na.drop()`** removes rows that contain null values (optionally restricted to specific
  columns via `subset=[...]`, or a threshold via `how="any"/"all"` and `thresh=`).
  It **reduces the row count**.
- **`.na.fill()`** replaces null values with a specified default (per-column or globally).
  It **keeps all rows**, just substitutes a value for the nulls.

Example — filling nulls in `status` with `'Unknown'`:

In [5]:
df_filled = df.na.fill({"status": "Unknown"})

df_filled.select("status").groupBy("status").count().show()


+--------+-----+
|  status|count|
+--------+-----+
|Inactive|  268|
|  Active|  253|
| Unknown|  287|
| Pending|  242|
+--------+-----+



## Q6: Count of records per city, only where count > 100

In [6]:
df_city_counts = (
    df.groupBy("city")
      .count()
      .filter(F.col("count") > 100)
      .orderBy(F.desc("count"))
)

df_city_counts.show()


+----+-----+
|city|count|
+----+-----+
+----+-----+



## Q7: Immutability of Spark DataFrames and data cleaning steps

Spark DataFrames are **immutable** — once created, a DataFrame's data cannot be changed in
place. Operations like dropping or renaming columns do **not** modify the original
DataFrame; instead, they return a **new DataFrame** with the transformation applied,
while the original DataFrame is left untouched.

Practical implications for data-cleaning code:

- You must **reassign the result** of a transformation (e.g., `df = df.drop("col")`) or
  store it under a new name (`df_clean = df.withColumnRenamed(...)`) — calling
  `df.drop("col")` alone has no effect unless you capture the return value.
- Because each transformation produces a new (logical) DataFrame, Spark can chain many
  cleaning steps together — `df.dropDuplicates().na.fill(...).withColumnRenamed(...)` —
  and build a single optimized execution plan (DAG) rather than mutating shared state,
  which also makes the pipeline safer for parallel/distributed execution.
- Immutability also enables **lineage tracking** — Spark knows exactly how each resulting
  DataFrame was derived, which supports fault tolerance (recomputing lost partitions) and
  `explain()`-able query plans.
- The trade-off is you must be careful with variable naming during multi-step cleaning, since
  it's easy to accidentally reference an older (pre-cleaning) version of the DataFrame.

## Q8: Filter age between 18 and 30 (inclusive) and subscription == 'Premium'

In [7]:
df_filtered = df.filter(
    (F.col("age").between(18, 30)) & (F.col("subscription") == "Premium")
)

print("Matching rows:", df_filtered.count())
df_filtered.select("user_id", "age", "subscription").show(10)


Matching rows: 78


+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
|  U1004| 20|     Premium|
|  U1007| 29|     Premium|
|  U1015| 27|     Premium|
|  U1024| 23|     Premium|
|  U1051| 30|     Premium|
|  U1060| 29|     Premium|
|  U1067| 24|     Premium|
|  U1079| 28|     Premium|
|  U1095| 21|     Premium|
|  U1102| 26|     Premium|
+-------+---+------------+
only showing top 10 rows


## Q9: Why handle nulls before aggregations like `sum()` / `avg()`?

- Spark's aggregation functions (`sum()`, `avg()`, etc.) **ignore nulls by default** —
  which sounds safe, but silently skewing a denominator or count can produce
  **misleading results** if nulls actually represent missing/invalid data rather than
  "nothing to add."
- If nulls are left in a numeric column, `avg()` divides by the count of **non-null**
  values only — this can overstate the average compared to what you'd get if the true
  (but missing) values were smaller, zero, or should have been excluded entirely.
- Unhandled nulls can also **propagate** into derived columns — an arithmetic
  expression involving a null (`price * quantity` where `quantity` is null) evaluates to
  null, silently dropping that row's contribution from any downstream aggregation.
- Handling nulls explicitly (deciding to `drop()` or `fill()` them, based on business logic)
  makes the aggregation's meaning **clear and reproducible** — the next person reading the
  pipeline can see exactly how missing data was treated, instead of relying on Spark's
  implicit null-skipping behavior.
- It also prevents **inconsistent results** across different aggregation functions in the
  same query, since some functions treat nulls differently depending on context.

## Q10: Cast `raw_timestamp` to `TimestampType` and rename to `event_time`

In [8]:
df_ts = df.withColumn("raw_timestamp", F.col("raw_timestamp").cast(TimestampType())) \
          .withColumnRenamed("raw_timestamp", "event_time")

df_ts.select("event_time").printSchema()
df_ts.select("event_time").show(5, truncate=False)


root
 |-- event_time: timestamp (nullable = true)



+-------------------+
|event_time         |
+-------------------+
|2025-01-27 21:47:00|
|2025-03-13 00:48:00|
|2025-07-06 14:34:00|
|2025-01-26 12:17:00|
|2025-03-11 20:44:00|
+-------------------+
only showing top 5 rows


## Q11: The "Shuffle" process in grouping operations

When you run a `groupBy()` followed by an aggregation (e.g., `groupBy("city").count()`),
Spark needs **all rows sharing the same key** (e.g., the same `city`) to end up on the
**same partition/executor** so they can be aggregated together.

Since the source data is typically spread arbitrarily across partitions (not grouped by key),
Spark must **redistribute data across the cluster** — rows are read from their original
partitions, hashed by the grouping key, and written/sent to new partitions where matching
keys land together. This process — moving data across the network between stages — is
called the **shuffle**.

**Why it's a wide transformation:** a transformation is "narrow" when each output partition
depends only on data from a single input partition (e.g., `filter()`, `map()`/`select()`) —
no data movement across partitions is needed. A transformation is "wide" when computing each
output partition requires data from **multiple** (potentially all) input partitions —
`groupBy`, `join`, `distinct`, and `repartition` are wide transformations because they
require this shuffle. Wide transformations are more expensive: they involve disk I/O
(writing shuffle files), network transfer between executors, and serialization/
deserialization, making them a common performance bottleneck in Spark jobs.

## Q12: Remove rows where `email` is null OR `username` is an empty string

In [9]:
df_clean_contacts = df.filter(
    ~(F.col("email").isNull() | (F.col("username") == ""))
)

print("Original rows:", df.count())
print("Rows after removing null email / empty username:", df_clean_contacts.count())
df_clean_contacts.select("user_id", "email", "username").show(5)


Original rows: 1050


Rows after removing null email / empty username: 957


+-------+-----------------+--------+
|user_id|            email|username|
+-------+-----------------+--------+
|  U1001|user1@example.com|  user_1|
|  U1002|user2@example.com|  user_2|
|  U1003|user3@example.com|  user_3|
|  U1004|user4@example.com|  user_4|
|  U1005|user5@example.com|  user_5|
+-------+-----------------+--------+
only showing top 5 rows


## Q13: Using `.agg()` to calculate multiple statistics at once

In [10]:
df_price_stats = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("mean_price"),
)

df_price_stats.show()


+---------+---------+------------------+
|min_price|max_price|        mean_price|
+---------+---------+------------------+
|     5.89|    498.7|250.74594069529655|
+---------+---------+------------------+



## Q14: Risk of `inferSchema=true` with messy/inconsistent date formats

When `inferSchema=true`, Spark samples the data and guesses a data type for each column
based on what it sees. This is risky for date columns when the source data has
**inconsistent formats** (e.g., mixing `2025-01-05`, `01/05/2025`, and `Jan 5, 2025` in the
same column):

- Spark's schema inference for dates expects a **single consistent pattern**. If the sampled
  rows happen to look consistent but other rows use a different format, Spark may infer the
  column as a **string** rather than a date/timestamp type — silently disabling any type-safe
  date operations later.
- Alternatively, if Spark does infer a date/timestamp type, rows that don't match the
  inferred format will be **silently converted to `null`** during parsing rather than raising
  an error — this can cause **silent data loss** that's easy to miss.
- Schema inference also requires an **extra pass over the data** (or a sample of it) purely
  to guess types, adding overhead compared to supplying an explicit schema.
- Because inference is based on a sample, it may not be **deterministic or reproducible**
  across different runs or different data sizes, especially near edge cases.

**Best practice:** explicitly define the schema (`StructType`) for a dataset with known,
messy date formats, and use `to_date()`/`to_timestamp()` with an explicit format string
(`F.to_date(F.col("raw_date"), "MM/dd/yyyy")`) so malformed values are handled predictably
rather than silently guessed at.

## Q15: Final processing pipeline

1. Filter out duplicates
2. Fill null prices with 0
3. Group by `store_id` to calculate total revenue

In [11]:
pipeline_df = (
    df.dropDuplicates(["user_id", "transaction_date"])   # 1. remove duplicates
      .na.fill({"price": 0})                              # 2. fill null prices with 0
      .groupBy("store_id")
      .agg(F.sum("price").alias("total_revenue"))          # 3. total revenue per store
      .orderBy(F.desc("total_revenue"))
)

pipeline_df.show()


+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
| STORE_5| 49651.66999999998|
| STORE_4|46890.359999999986|
| STORE_3|          46598.63|
| STORE_1| 45977.55000000003|
| STORE_2| 43715.47999999999|
+--------+------------------+



In [12]:
# Save the final pipeline result to the output folder as required by the assignment guide
pipeline_pd = pipeline_df.toPandas()
pipeline_pd.to_csv("../output/results.csv", index=False)
print("Saved results.csv with", len(pipeline_pd), "rows")
pipeline_pd


/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Saved results.csv with 5 rows


,store_id,total_revenue
0,STORE_5,49651.67
1,STORE_4,46890.36
2,STORE_3,46598.63
3,STORE_1,45977.55
4,STORE_2,43715.48


## Insights / Summary

- **Cleaning:** the raw dataset contained duplicate `(user_id, transaction_date)` pairs,
  null `status`/`price` values, and empty `username`/`email` fields — all of which were
  handled with `dropDuplicates()`, `.na.fill()`, and boolean filters before any aggregation.
- **Filtering:** combining multiple conditions (e.g., age range **and** subscription tier)
  with `&`/`|` operators lets Spark push filters down efficiently before wider operations.
- **Aggregation:** `.agg()` with several functions in one call (`min`, `max`, `mean`) avoids
  triggering multiple separate jobs compared to calling each stat individually.
- **Grouping/shuffle:** `groupBy()` operations (city counts, category averages, per-store
  revenue) are wide transformations that trigger a shuffle — worth minimizing on very large
  datasets by filtering early and only grouping on what's needed.
- **Schema handling:** casting `raw_timestamp` to `TimestampType` and renaming it to
  `event_time` shows how immutable transformations return new DataFrames rather than
  mutating in place — every step in this notebook reassigns the result to a new variable.
- **Final pipeline (Q15):** chaining dedup → null-fill → groupBy/agg demonstrates a
  realistic end-to-end Spark cleaning-and-aggregation workflow, and its output was saved to
  `output/results.csv` per the assignment's suggested folder structure.


In [13]:
spark.stop()